In [1]:
import pandas as pd
import numpy as np
from collections import defaultdict

def load_data():
    """Load and clean the data"""
    projects = pd.read_csv('pb_projects.csv')
    votes = pd.read_csv('pb_votes.csv')
    
    votes['Projects Selected'] = votes['Projects Selected'].str.split(',')
    votes['Projects Selected'] = votes['Projects Selected'].apply(
        lambda x: [p.strip() for p in x if p.strip()]
    )
    
    return projects, votes

def initialize_mes(projects, votes, total_budget):
    """Prepare data structures for MES"""
   
    voter_share = total_budget / len(votes)
    
    project_supporters = defaultdict(list)
    voter_budgets = {}
    voter_contributions = {}
    
    for _, row in votes.iterrows():
        voter = row['Voter ID']
        voter_budgets[voter] = voter_share
        voter_contributions[voter] = {}
        
        for project in row['Projects Selected']:
            project_supporters[project].append(voter)
    
    return project_supporters, voter_budgets, voter_contributions, voter_share

def mes_algorithm(projects, project_supporters, voter_budgets, voter_contributions, voter_share):
    """Core MES implementation"""
    funded_projects = []
    project_costs = projects.set_index('Project ID')['Cost (₹ crore)'].to_dict()
    
    while True:
        affordable_projects = []
        
        for project in set(project_costs.keys()) - set(funded_projects):
            cost = project_costs[project]
            supporters = project_supporters.get(project, [])
            
            if not supporters:
                continue
            
            low, high = 0, 1
            for _ in range(20):
                alpha = (low + high) / 2
                total = sum(min(voter_budgets[v], alpha * voter_share) for v in supporters)
                if total < cost:
                    low = alpha
                else:
                    high = alpha
            
            alpha = high
            total_potential = sum(min(voter_budgets[v], alpha * voter_share) for v in supporters)
            if total_potential >= cost:
                affordable_projects.append({
                    'project': project,
                    'cost': cost,
                    'supporters': supporters,
                    'alpha': alpha,
                    'votes': len(supporters)
                })
        
        if not affordable_projects:
            break
            
        affordable_projects.sort(
            key=lambda x: (-x['alpha'], -x['votes'] / x['cost'])
        )
        selected = affordable_projects[0]
        
        payment_per_voter = selected['alpha'] * voter_share
        
        for voter in selected['supporters']:
            payment = min(voter_budgets[voter], payment_per_voter)
            voter_budgets[voter] -= payment
            voter_contributions[voter][selected['project']] = payment
        
        funded_projects.append(selected['project'])
    
    return funded_projects, voter_budgets

def analyze_results(projects, funded_projects, voter_contributions, voter_budgets):
    """Generate comprehensive results report"""
    funded_df = projects[projects['Project ID'].isin(funded_projects)].copy()
    funded_df['Votes'] = funded_df['Project ID'].apply(
        lambda x: sum(1 for v in voter_contributions.values() if x in v)
    )
    
    remaining_budget = sum(voter_budgets.values())
    total_spent = funded_df['Cost (₹ crore)'].sum()
    
    print(f"\n{'='*60}")
    print(f"FUNDED: {len(funded_df)} projects (₹{total_spent:.2f} crore)")
    print(f"REMAINING BUDGET: ₹{remaining_budget:.2f} crore")
    print("="*60)
    
    print("\nREGIONAL DISTRIBUTION:")
    print(funded_df['Region'].value_counts())
    
    print("\nCATEGORY DISTRIBUTION:")
    print(funded_df['Category'].value_counts())
    
    print("\nTOP FUNDED PROJECTS:")
    print(funded_df[['Project ID', 'Project Name', 'Cost (₹ crore)', 'Votes']]
          .sort_values('Votes', ascending=False)
          .head(5).to_string(index=False))
    
    return funded_df

def save_results(funded_df, projects):
    """Save results to CSV"""
    unfunded_df = projects[~projects['Project ID'].isin(funded_df['Project ID'])]
    
    funded_df.to_csv('funded_projects_mes.csv', index=False)
    unfunded_df.to_csv('unfunded_projects_mes.csv', index=False)
    print("\n✅ Results saved to funded_projects_mes.csv and unfunded_projects_mes.csv")

def main():
    TOTAL_BUDGET = 500  # ₹500 crore
    
    projects, votes = load_data()
    
    project_supporters, voter_budgets, voter_contributions, voter_share = initialize_mes(
        projects, votes, TOTAL_BUDGET
    )
    
    funded_projects, voter_budgets = mes_algorithm(
        projects, project_supporters, voter_budgets, voter_contributions, voter_share
    )
    
    funded_df = analyze_results(projects, funded_projects, voter_contributions, voter_budgets)
    
    save_results(funded_df, projects)

if __name__ == "__main__":
    main()



FUNDED: 11 projects (₹452.00 crore)
REMAINING BUDGET: ₹48.00 crore

REGIONAL DISTRIBUTION:
Region
Mumbai         2
Marathwada     2
Pune           1
Konkan         1
Gadchiroli     1
Aurangabad     1
Kolhapur       1
Latur          1
Navi Mumbai    1
Name: count, dtype: int64

CATEGORY DISTRIBUTION:
Category
Transport         2
Water             2
Environment       2
Agriculture       1
Education         1
Employment        1
Industry          1
Infrastructure    1
Name: count, dtype: int64

TOP FUNDED PROJECTS:
Project ID                Project Name  Cost (₹ crore)  Votes
        P9           Aurangabad IT Hub              60   1937
       P15   Navi Mumbai Airport Roads              75   1911
        P3   Pune Water Supply Upgrade              45   1905
        P7      Tribal School Upgrades              20   1900
        P5 Farmer Solar Pump Subsidies              25   1898

✅ Results saved to funded_projects_mes.csv and unfunded_projects_mes.csv


In [11]:
import pandas as pd
from collections import defaultdict

def load_data():
    """Load projects and voters data"""
    projects = pd.read_csv('pb_projects.csv')
    votes = pd.read_csv('pb_votes.csv')
    
    votes['Projects Selected'] = votes['Projects Selected'].str.split(',')
    votes['Projects Selected'] = votes['Projects Selected'].apply(
        lambda x: [p.strip() for p in x if p.strip()]
    )
    
    return projects, votes

def calculate_votes(votes_data):
    """Count votes for each project"""
    project_votes = defaultdict(int)
    for _, row in votes_data.iterrows():
        for project in row['Projects Selected']:
            project_votes[project] += 1
    return project_votes

def knapsack_algorithm(projects, project_votes, total_budget):
    """0/1 Knapsack implementation for project selection"""
    project_list = []
    for _, proj in projects.iterrows():
        pid = proj['Project ID']
        if pid in project_votes:
            project_list.append({
                'id': pid,
                'votes': project_votes[pid],
                'cost': proj['Cost (₹ crore)'],
                'name': proj['Project Name'],
                'region': proj['Region'],
                'category': proj['Category']
            })
    
    n = len(project_list)
    budget = int(total_budget * 100) 
    
    dp = [[0] * (budget + 1) for _ in range(n + 1)]
    
    for i in range(1, n + 1):
        cost = int(project_list[i-1]['cost'] * 100)
        votes = project_list[i-1]['votes']
        
        for w in range(1, budget + 1):
            if cost <= w:
                dp[i][w] = max(dp[i-1][w], dp[i-1][w-cost] + votes)
            else:
                dp[i][w] = dp[i-1][w]
    
    w = budget
    selected_projects = []
    
    for i in range(n, 0, -1):
        if dp[i][w] != dp[i-1][w]:
            selected_projects.append(project_list[i-1]['id'])
            w -= int(project_list[i-1]['cost'] * 100)
    
    funded_df = pd.DataFrame([p for p in project_list if p['id'] in selected_projects])
    total_cost = funded_df['cost'].sum()
    total_votes = funded_df['votes'].sum()
    
    return funded_df, total_cost, total_votes

def generate_report(funded_df, total_cost, total_votes, original_projects):
    """Generate results report"""
    unfunded_df = original_projects[~original_projects['Project ID'].isin(funded_df['id'])]
    
    print(f"\n{'='*60}")
    print(f"FUNDED: {len(funded_df)} projects (₹{total_cost:.2f} crore)")
    print(f"TOTAL VOTES: {total_votes}")
    print(f"UNFUNDED: {len(unfunded_df)} projects")
    print("="*60)
    
    print("\nREGIONAL DISTRIBUTION:")
    print(funded_df['region'].value_counts())
    
    print("\nCATEGORY DISTRIBUTION:")
    print(funded_df['category'].value_counts())
    
    print("\nTOP FUNDED PROJECTS:")
    print(funded_df[['id', 'name', 'cost', 'votes']]
          .sort_values('votes', ascending=False)
          .head(5).to_string(index=False))
    
    return unfunded_df

def save_results(funded_df, unfunded_df):
    """Save results to CSV"""
    funded_df.rename(columns={
        'id': 'Project ID',
        'name': 'Project Name',
        'cost': 'Cost (₹ crore)',
        'votes': 'Votes'
    }).to_csv('funded_projects_knapsack.csv', index=False)
    
    unfunded_df.to_csv('unfunded_projects_knapsack.csv', index=False)
    print("\nResults saved to funded_projects_knapsack.csv and unfunded_projects_knapsack.csv")

def main():
    TOTAL_BUDGET = 500  # ₹500 crore
    
    projects, votes = load_data()
    project_votes = calculate_votes(votes)
    
    funded_df, total_cost, total_votes = knapsack_algorithm(
        projects, project_votes, TOTAL_BUDGET
    )
    
    unfunded_df = generate_report(
        funded_df, total_cost, total_votes, projects
    )
    
    save_results(funded_df, unfunded_df)

if __name__ == "__main__":
    main()


FUNDED: 16 projects (₹497.00 crore)
TOTAL VOTES: 30364
UNFUNDED: 5 projects

REGIONAL DISTRIBUTION:
region
Vidarbha      2
Pune          2
Marathwada    2
Mumbai        2
Nashik        2
Konkan        1
Gadchiroli    1
Aurangabad    1
Thane         1
Latur         1
Nagpur        1
Name: count, dtype: int64

CATEGORY DISTRIBUTION:
category
Environment       3
Water             2
Education         2
Employment        2
Tourism           2
Infrastructure    2
Healthcare        1
Agriculture       1
Sanitation        1
Name: count, dtype: int64

TOP FUNDED PROJECTS:
 id                               name  cost  votes
P17             Green Belt Development    26   1964
 P9                  Aurangabad IT Hub    60   1937
P21        Cultural Center Restoration    19   1923
P18           Skill Training for Youth    33   1915
P16 Smart Classrooms for Rural Schools    22   1913

Results saved to funded_projects_knapsack.csv and unfunded_projects_knapsack.csv


In [12]:
import pandas as pd
from collections import defaultdict

def load_data():
    """Load projects and voters data with correct file names"""
    projects = pd.read_csv('pb_projects.csv')
    votes = pd.read_csv('pb_votes.csv')
    
    votes['Projects Selected'] = votes['Projects Selected'].str.split(',')
    votes['Projects Selected'] = votes['Projects Selected'].apply(
        lambda x: [p.strip() for p in x if p.strip()]  # Remove whitespace and empty strings
    )
    
    return projects, votes

def calculate_votes(votes_data):
    """Count votes for each project"""
    project_votes = defaultdict(int)
    for _, row in votes_data.iterrows():
        for project in row['Projects Selected']:
            project_votes[project] += 1
    return project_votes

def greedy_allocation(projects, project_votes, total_budget):
    """Greedy algorithm implementation"""
    project_values = []
    for _, proj in projects.iterrows():
        pid = proj['Project ID']
        if pid in project_votes:
            value = project_votes[pid] / proj['Cost (₹ crore)']
            project_values.append({
                'Project ID': pid,
                'Value': value,
                'Votes': project_votes[pid],
                'Cost': proj['Cost (₹ crore)'],
                'Project Name': proj['Project Name'],
                'Region': proj['Region'],
                'Category': proj['Category']
            })
    
    project_values.sort(key=lambda x: x['Value'], reverse=True)
    
    funded = []
    remaining = total_budget
    
    for proj in project_values:
        if proj['Cost'] <= remaining:
            funded.append(proj)
            remaining -= proj['Cost']
    
    return funded, remaining

def generate_report(funded_projects, remaining_budget, original_projects):
    """Generate comprehensive results report"""
    funded_df = pd.DataFrame(funded_projects)
    
    funded_ids = [p['Project ID'] for p in funded_projects]
    unfunded_df = original_projects[~original_projects['Project ID'].isin(funded_ids)]
    
    print(f"\n{'='*60}")
    print(f"FUNDED: {len(funded_df)} projects (₹{funded_df['Cost'].sum():.2f} crore)")
    print(f"UNFUNDED: {len(unfunded_df)} projects")
    print(f"REMAINING BUDGET: ₹{remaining_budget:.2f} crore")
    print("="*60)
    
    print("\nREGIONAL DISTRIBUTION:")
    print(funded_df['Region'].value_counts())
    
    print("\nCATEGORY DISTRIBUTION:")
    print(funded_df['Category'].value_counts())
    
    print("\nTOP 5 PROJECTS BY VALUE:")
    print(funded_df[['Project ID', 'Project Name', 'Cost', 'Votes', 'Value']]
          .head(5).to_string(index=False))
    
    return funded_df, unfunded_df

def save_results(funded, unfunded):
    """Save results to CSV files"""
    funded.to_csv('funded_projects_greedy.csv', index=False)
    unfunded.to_csv('unfunded_projects_greedy.csv', index=False)
    print("\nResults saved to: funded_projects_greedy.csv and unfunded_projects_greedy.csv")

def main():
    TOTAL_BUDGET = 500  # ₹500 crore
    
    projects, votes = load_data()
    project_votes = calculate_votes(votes)
    
    funded_projects, remaining = greedy_allocation(projects, project_votes, TOTAL_BUDGET)
    
    funded_df, unfunded_df = generate_report(funded_projects, remaining, projects)
    
    save_results(funded_df, unfunded_df)

if __name__ == "__main__":
    main()


FUNDED: 16 projects (₹455.00 crore)
UNFUNDED: 5 projects
REMAINING BUDGET: ₹45.00 crore

REGIONAL DISTRIBUTION:
Region
Mumbai        2
Nashik        2
Vidarbha      2
Marathwada    2
Pune          2
Konkan        1
Amravati      1
Gadchiroli    1
Latur         1
Nagpur        1
Thane         1
Name: count, dtype: int64

CATEGORY DISTRIBUTION:
Category
Environment       3
Tourism           2
Education         2
Water             2
Infrastructure    2
Social Welfare    1
Agriculture       1
Healthcare        1
Employment        1
Sanitation        1
Name: count, dtype: int64

TOP 5 PROJECTS BY VALUE:
Project ID                Project Name  Cost  Votes      Value
       P20       Urban Tree Plantation    12   1850 154.166667
        P6     Coastal Erosion Control    15   1886 125.733333
       P11    Amravati Women’s Hostels    18   1856 103.111111
       P21 Cultural Center Restoration    19   1923 101.210526
        P7      Tribal School Upgrades    20   1900  95.000000

Results saved 

In [2]:
import pandas as pd
from collections import defaultdict

def load_data():
    """Load and clean the data"""
    projects = pd.read_csv('pb_projects.csv')
    votes = pd.read_csv('pb_votes.csv')
    
    votes['Projects Selected'] = votes['Projects Selected'].str.split(',')
    votes['Projects Selected'] = votes['Projects Selected'].apply(
        lambda x: [p.strip() for p in x if p.strip()]
    )
    
    return projects, votes

def initialize_mes(projects, votes, total_budget):
    """Prepare data structures for MES"""
    voter_share = total_budget / len(votes)
    project_supporters = defaultdict(list)
    voter_budgets = {}
    voter_contributions = {}
    
    for _, row in votes.iterrows():
        voter = row['Voter ID']
        voter_budgets[voter] = voter_share
        voter_contributions[voter] = {}
        
        for project in row['Projects Selected']:
            project_supporters[project].append(voter)
    
    return project_supporters, voter_budgets, voter_contributions, voter_share

def mes_phase(projects, project_supporters, voter_budgets, voter_contributions, voter_share):
    """Core MES implementation"""
    funded_projects = []
    project_costs = projects.set_index('Project ID')['Cost (₹ crore)'].to_dict()
    
    while True:
        affordable_projects = []
        
        for project in set(project_costs.keys()) - set(funded_projects):
            cost = project_costs[project]
            supporters = project_supporters.get(project, [])
            
            if not supporters:
                continue
            
            low, high = 0, 1
            for _ in range(20):
                alpha = (low + high) / 2
                total = sum(min(voter_budgets[v], alpha * voter_share) for v in supporters)
                if total < cost:
                    low = alpha
                else:
                    high = alpha
            
            alpha = high
            total_potential = sum(min(voter_budgets[v], alpha * voter_share) for v in supporters)
            if total_potential >= cost:
                affordable_projects.append({
                    'project': project,
                    'cost': cost,
                    'supporters': supporters,
                    'alpha': alpha,
                    'votes': len(supporters)
                })
        
        if not affordable_projects:
            break
            
        affordable_projects.sort(key=lambda x: (-x['alpha'], -x['votes'] / x['cost']))
        selected = affordable_projects[0]
        payment_per_voter = selected['alpha'] * voter_share
        
        for voter in selected['supporters']:
            payment = min(voter_budgets[voter], payment_per_voter)
            voter_budgets[voter] -= payment
            voter_contributions[voter][selected['project']] = payment
        
        funded_projects.append(selected['project'])
    
    return funded_projects, voter_budgets

def knapsack_completion(projects, funded_projects, voter_budgets, project_supporters, total_budget):
    """Knapsack-style allocation of remaining funds"""
    remaining_budget = sum(voter_budgets.values())
    if remaining_budget <= 0:
        return funded_projects
    
    avg_voter_budget = total_budget / len(voter_budgets)
    underrepresented = {
        v for v, budget in voter_budgets.items() 
        if budget > 0.5 * avg_voter_budget
    }
    
    project_scores = []
    project_costs = projects.set_index('Project ID')['Cost (₹ crore)'].to_dict()
    
    for project in set(projects['Project ID']) - set(funded_projects):
        supporters = project_supporters.get(project, [])
        under_support = len([v for v in supporters if v in underrepresented])
        cohesion = under_support / len(supporters) if supporters else 0
        
        if under_support > 0:
            score = (under_support / project_costs[project]) * cohesion
            project_scores.append({
                'project': project,
                'score': score,
                'cost': project_costs[project]
            })
    
    project_scores.sort(key=lambda x: -x['score'])
    
    for proj in project_scores:
        if proj['cost'] <= remaining_budget:
            funded_projects.append(proj['project'])
            remaining_budget -= proj['cost']
    
    return funded_projects

def analyze_results(projects, funded_projects, voter_contributions, total_budget):
    """Generate comprehensive results report"""
    funded_df = projects[projects['Project ID'].isin(funded_projects)].copy()
    unfunded_df = projects[~projects['Project ID'].isin(funded_projects)]
    
    funded_df['Votes'] = funded_df['Project ID'].apply(
        lambda x: sum(1 for v in voter_contributions.values() if x in v)
    )
    
    total_spent = funded_df['Cost (₹ crore)'].sum()
    remaining_budget = total_budget - total_spent

    print(f"\n{'='*60}")
    print(f"FUNDED: {len(funded_df)} projects (₹{total_spent:.2f} crore)")
    print(f"UNFUNDED: {len(unfunded_df)} projects")
    print(f"REMAINING BUDGET: ₹{remaining_budget:.2f} crore")
    print("="*60)

    print("\nREGIONAL DISTRIBUTION (FUNDED):")
    print(funded_df['Region'].value_counts())

    print("\nCATEGORY DISTRIBUTION (FUNDED):")
    print(funded_df['Category'].value_counts())

    print("\nTOP FUNDED PROJECTS:")
    print(funded_df[['Project ID', 'Project Name', 'Cost (₹ crore)', 'Votes']]
          .sort_values('Votes', ascending=False)
          .head(5).to_string(index=False))
    
    print("\nLARGEST UNFUNDED PROJECTS:")
    print(unfunded_df[['Project ID', 'Project Name', 'Cost (₹ crore)']]
          .sort_values('Cost (₹ crore)', ascending=False)
          .head(5).to_string(index=False))

    return funded_df, unfunded_df

def save_results(funded_df, unfunded_df):
    """Save results to CSV files"""
    funded_df.to_csv('funded_projects_hybrid.csv', index=False)
    unfunded_df.to_csv('unfunded_projects_hybrid.csv', index=False)
    print("\n✅ Results saved to:")
    print("- funded_projects_hybrid.csv")
    print("- unfunded_projects_hybrid.csv")

def main():
    total_budget = 500  # ₹500 crore
    
    projects, votes = load_data()
    
    project_supporters, voter_budgets, voter_contributions, voter_share = initialize_mes(
        projects, votes, total_budget
    )
    
    funded_projects, voter_budgets = mes_phase(
        projects, project_supporters, voter_budgets, voter_contributions, voter_share
    )
    
    funded_projects = knapsack_completion(
        projects, funded_projects, voter_budgets, project_supporters, total_budget
    )
    
    funded_df, unfunded_df = analyze_results(projects, funded_projects, voter_contributions, total_budget)
    save_results(funded_df, unfunded_df)

if __name__ == "__main__":
    main()


FUNDED: 13 projects (₹489.00 crore)
UNFUNDED: 8 projects
REMAINING BUDGET: ₹11.00 crore

REGIONAL DISTRIBUTION (FUNDED):
Region
Mumbai         2
Marathwada     2
Pune           1
Konkan         1
Gadchiroli     1
Aurangabad     1
Amravati       1
Kolhapur       1
Latur          1
Navi Mumbai    1
Nashik         1
Name: count, dtype: int64

CATEGORY DISTRIBUTION (FUNDED):
Category
Transport         2
Water             2
Environment       2
Agriculture       1
Education         1
Employment        1
Social Welfare    1
Industry          1
Infrastructure    1
Tourism           1
Name: count, dtype: int64

TOP FUNDED PROJECTS:
Project ID                Project Name  Cost (₹ crore)  Votes
        P9           Aurangabad IT Hub              60   1937
       P15   Navi Mumbai Airport Roads              75   1911
        P3   Pune Water Supply Upgrade              45   1905
        P7      Tribal School Upgrades              20   1900
        P5 Farmer Solar Pump Subsidies              25   1